In [2]:
# ============================================================================
# STEP 1: ENVIRONMENT SETUP & LIBRARY INSTALLATION
# ============================================================================

%pip install sentence-transformers scikit-learn pandas numpy tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: C:\Users\rg710\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [3]:
# ============================================================================
# STEP 2: INITIALIZE MODEL AND WORKSPACE PATHS
# ============================================================================

import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Configure explicit storage system paths
full_dataset_path = 'D:/[PUB] India_runs_data_and_ai_challenge/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/candidates.jsonl'
output_submission_path = 'team_ravi_submission.csv'

print("Loading semantic embedding model (BAAI/bge-large-en-v1.5)...")
model = SentenceTransformer('BAAI/bge-large-en-v1.5')

# Core description requirements used to calculate AI similarity scores
job_description_text = """
Senior AI Engineer — Founding Team at Redrob AI. 
Deep technical depth in modern ML systems: embeddings, retrieval, ranking, LLMs, fine-tuning, RAG system architecture, vector search.
Scrappy product-engineering attitude, building scalable recommendation systems.
Location: Pune or Noida, India. Experience: 5 to 9 years.
"""

print("Encoding job description vector...")
job_embedding = model.encode([job_description_text])
print("Workspace configurations completed!")

Loading semantic embedding model (BAAI/bge-large-en-v1.5)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Encoding job description vector...
Workspace configurations completed!


In [4]:
# ============================================================================
# STEP 3: STRATEGIC EVALUATION & BEHAVIORAL SIGNALS PARSER
# ============================================================================

def evaluate_candidates(df):
    processed_results = []
    
    for idx, row in df.iterrows():
        cid = row['candidate_id']
        title = str(row.get('profile.current_title', '')).lower()
        summary = str(row.get('profile.summary', '')).lower()
        headline = str(row.get('profile.headline', '')).lower()
        
        # 1. Neutralize Keyword Stuffing Traps
        invalid_keywords = ['marketing', 'sales', 'hr manager', 'graphic designer', 'content writer', 'recruiter', 'mechanical engineer']
        is_trap = any(word in title for word in invalid_keywords)
        title_multiplier = 0.05 if is_trap else 1.0
        
        # 2. Calculate Target Experience Boundaries (Target: 5 to 9 Years)
        exp_years = row.get('profile.years_of_experience', 0)
        if pd.isna(exp_years): exp_years = 0
            
        if 5 <= exp_years <= 9:
            exp_score = 1.0
        elif 4 <= exp_years < 5 or 9 < exp_years <= 11:
            exp_score = 0.5
        else:
            exp_score = 0.1
            
        # 3. Analyze Geographical/Relocation Demographics
        loc = str(row.get('profile.location', '')).lower()
        country = str(row.get('profile.country', '')).lower()
        willing_to_relocate = row.get('redrob_signals.willing_to_relocate', False)
        
        loc_score = 0.5
        if 'pune' in loc or 'noida' in loc:
            loc_score = 1.0
        elif willing_to_relocate and ('india' in country or 'india' in loc):
            loc_score = 0.9
            
        # 4. Integrate Platform Activity Availability Multipliers
        response_rate = row.get('redrob_signals.recruiter_response_rate', 0.5)
        if pd.isna(response_rate): response_rate = 0.5
            
        interview_rate = row.get('redrob_signals.interview_completion_rate', 0.5)
        if pd.isna(interview_rate): interview_rate = 0.5
            
        behavioral_multiplier = 0.4 + (response_rate * 0.3) + (interview_rate * 0.3)
        
        # 5. Compile Global Dense Context Representation
        skills_list = row.get('skills', [])
        skills_text = ", ".join([s['name'] for s in skills_list if isinstance(s, dict) and 'name' in s]) if isinstance(skills_list, list) else ""
        combined_text = f"Title: {title}. Headline: {headline}. Summary: {summary}. Skills: {skills_text}."
        
        processed_results.append({
            'candidate_id': cid,
            'combined_text': combined_text,
            'title_multiplier': title_multiplier,
            'exp_score': exp_score,
            'loc_score': loc_score,
            'behavioral_multiplier': behavioral_multiplier,
            'current_title': title,
            'years_of_experience': exp_years
        })
        
    return pd.DataFrame(processed_results)
print("Scoring matrix compiled successfully!")

Scoring matrix compiled successfully!


In [6]:
# ============================================================================
# STEP 4: MEMORY-OPTIMIZED STREAMING AND AGGRESSIVE PRE-FILTERING
# ============================================================================

print("Streaming and filtering based on highly strict constraints...")
valid_candidates = []

# Core technical title requirements to dramatically speed up embedding runtimes
strict_ai_keywords = ['ai', 'machine learning', 'ml', 'deep learning', 'nlp', 'data scientist', 'computer vision']

with open(full_dataset_path, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Streaming Dataset Pool"):
        line = line.strip()
        if not line:
            continue
            
        candidate = json.loads(line)
        
        title = str(candidate.get('profile', {}).get('current_title', '')).lower()
        exp_years = candidate.get('profile', {}).get('years_of_experience', 0)
        loc = str(candidate.get('profile', {}).get('location', '')).lower()
        country = str(candidate.get('profile', {}).get('country', '')).lower()
        willing_to_relocate = candidate.get('redrob_signals', {}).get('willing_to_relocate', False)
        
        # Rule A: Strict Title Filtering (Must be an explicit AI/ML profile)
        if not any(keyword in title for keyword in strict_ai_keywords):
            continue
            
        # Rule B: Evade Honeypot traps by dropping non-technical designations
        invalid_keywords = ['marketing', 'sales', 'hr manager', 'graphic designer', 'content writer', 'recruiter']
        if any(word in title for word in invalid_keywords):
            continue
            
        # Rule C: Narrow experience window envelope check (Strict 5-10 years)
        if exp_years < 5 or exp_years > 10:
            continue
            
        # Rule D: Target region relocation alignment (Pune or Noida only)
        is_target_loc = 'pune' in loc or 'noida' in loc
        is_india_relocate = willing_to_relocate and ('india' in country or 'india' in loc)
        if not (is_target_loc or is_india_relocate):
            continue
            
        valid_candidates.append(candidate)

print(f"\nSuccessfully compressed pool from 100K down to {len(valid_candidates)} premium target profiles!")

Streaming and filtering based on highly strict constraints...


Streaming Dataset Pool: 100000it [00:06, 16257.21it/s]


Successfully compressed pool from 100K down to 252 premium target profiles!


In [7]:
# ============================================================================
# STEP 5: COMPUTE AI EMBEDDINGS, SORT TIES AND EXPORT SUBMISSION CSV
# ============================================================================

if len(valid_candidates) > 0:
    # Normalize the filtered JSON data into a clean Pandas DataFrame
    df_filtered = pd.json_normalize(valid_candidates)
    df_proc = evaluate_candidates(df_filtered)
    
    print("Generating vector spatial mappings for the target subset pool...")
    texts = df_proc['combined_text'].tolist()
    
    # Execution completes in seconds because the input list is pre-filtered
    embeddings = model.encode(texts, batch_size=32, show_progress_bar=True)
    scores = cosine_similarity(job_embedding, embeddings)[0]
    
    # Assign the calculated semantic score
    df_proc['semantic_score'] = scores
    
    # Apply our multi-layered strategic scoring formula
    df_proc['final_score'] = (
        df_proc['semantic_score'] * df_proc['title_multiplier'] * df_proc['exp_score'] * df_proc['loc_score'] * df_proc['behavioral_multiplier']
    )
    
    # Compliance Rule: Sort by score descending. Tie-breaker: candidate_id alphanumeric ascending.
    df_proc = df_proc.sort_values(by=['final_score', 'candidate_id'], ascending=[False, True]).reset_index(drop=True)
    
    # Extract the exact top 100 entries required by submission_spec.docx
    df_top100 = df_proc.head(100).copy()
    df_top100['rank'] = df_top100.index + 1
    
    # Generate the requested structured reasoning tracking strings
    df_top100['reasoning'] = df_top100.apply(
        lambda r: f"AI Systems Engineer with {r['years_of_experience']} years of experience. Specialist in modern ML infrastructure. Current title: {r['current_title']}.", 
        axis=1
    )
    
    # Format and conform precisely to standard verification schema layout rules
    submission_format = df_top100[['candidate_id', 'rank', 'final_score', 'reasoning']].rename(columns={'final_score': 'score'})
    
    # Save the absolute final file to your local drive path
    submission_format.to_csv(output_submission_path, index=False)
    
    print(f"\n>>> Success! Official hackathon file created at: {output_submission_path} <<<")
    print("\nPreview of your Top 5 Candidate Rankings:")
    display(submission_format.head(5))
else:
    print("Error: Pipeline subset execution array contains zero elements. Please re-run Cell 4.")

Generating vector spatial mappings for the target subset pool...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]


>>> Success! Official hackathon file created at: team_ravi_submission.csv <<<

Preview of your Top 5 Candidate Rankings:


,candidate_id,rank,score,reasoning
0,CAND_0046525,1,0.733095,AI Systems Engineer with 6.1 years of experien...
1,CAND_0053605,2,0.717965,AI Systems Engineer with 6.9 years of experien...
2,CAND_0093763,3,0.713833,AI Systems Engineer with 5.0 years of experien...
3,CAND_0079064,4,0.709534,AI Systems Engineer with 5.2 years of experien...
4,CAND_0006567,5,0.706756,AI Systems Engineer with 7.9 years of experien...
